## 📅 Day 07 — Aggregation Utilities & Data Quality Metrics
### 🌍 Real-World Scenario (Energy Analytics)
You’re now in an energy analytics team.

Stakeholders don’t want:
    raw records
    cleaned lists

They want: 
    1.metrics
    2.summaries
    3.signals they can act on


**Examples:**
    total energy demand
    demand by site
    peak vs off-peak usage
    percentage of invalid data



### 🎯 Day 07 Goal
By the end of today, you will:

    build aggregation utilities
    
    compute data quality metrics
    
    design functions that produce business-ready summaries
    
    connect clean data → insights

### 🧠 Concepts You’ll Practice
    Aggregation patterns (sum, count, group-by)
    
    Metric design
    
    Separation of raw data vs derived insights
    
    Writing analysis-ready utilities


### 🔹 Task 1 — Aggregate Demand by Site
Problem

“How much energy demand comes from each site?”

In [2]:
energy_records = [
    {"site_id": "S1", "timestamp": "2024-01-01 08:00", "demand_kw": 120.0},
    {"site_id": "S1", "timestamp": "2024-01-01 09:00", "demand_kw": 150.0},
    {"site_id": "S2", "timestamp": "2024-01-01 08:00", "demand_kw": 60.0},
    {"site_id": "S2", "timestamp": "2024-01-01 18:00", "demand_kw": 90.0},
    {"site_id": "S3", "timestamp": "2024-01-01 13:00", "demand_kw": 200.0},
    {"site_id": "S1", "timestamp": "2024-01-01 18:00", "demand_kw": 80.0},
    {"site_id": "S3", "timestamp": "2024-01-01 19:00", "demand_kw": 40.0},
]


In [1]:
def aggregate_demand_by_site( energy_records):
    """ aggregate total demand per site
    """
    aggregate_energy_demand_by_site = {}
    for record in energy_records:
        demand = record['demand_kw']
        site_id = record['site_id']
        
        if site_id not in aggregate_energy_demand_by_site:
            aggregate_energy_demand_by_site[site_id] = 0.0

        aggregate_energy_demand_by_site[site_id] += demand
        
    return aggregate_energy_demand_by_site
  

In [3]:
aggregate_demand_by_site( energy_records)

{'S1': 350.0, 'S2': 150.0, 'S3': 240.0}

### 🔹 Task 2 — Peak vs Off-Peak Demand Summary
Problem

Energy planning often depends on peak usage.

In [5]:
records = [
    {"site_id": "S1", "timestamp": "2024-01-01 08:00", "demand_kw": 120.0, "is_peak": False},
    {"site_id": "S1", "timestamp": "2024-01-01 09:00", "demand_kw": 150.0, "is_peak": False},
    {"site_id": "S2", "timestamp": "2024-01-01 18:00", "demand_kw": 90.0,  "is_peak": True},
    {"site_id": "S3", "timestamp": "2024-01-01 19:00", "demand_kw": 40.0,  "is_peak": True},
    {"site_id": "S3", "timestamp": "2024-01-01 13:00", "demand_kw": 200.0, "is_peak": False},
]

In [8]:
def summarize_peak_or_peak_off(records):
    """Return total demand during peak and peak off hours.
    """
    summary = {
        'peak':0.0,
        'off_peak':0.0
    }
    for record in records:
        is_peak = record['is_peak']
        demand = record['demand_kw']
        
        if is_peak :
            summary['peak'] += demand
        else :
            summary['off_peak'] += demand
            
    return summary           

In [9]:
summerize_peak_and_peak_off(records)

{'peak': 130.0, 'off_peak': 470.0}

### 🔹 Task 3 — Data Quality Metrics (Very Important)
Problem :  You want to report:

how clean your data is

how much was rejected

In [10]:
# from Day 05 I have valid and invalid records : valid_records, invalid_records = validate_energy_batch(records)


In [17]:
def data_quality_metrics( total_records , invalid_records):
    """
    compute data quality
    """
    total = len(total_records)
    invalid =  len(invalid_records)
    valid = total - invalid
    # computing invalid percentage
    invalid_percentage = round((invalid/total )*100 ,2)   if total >0 else 0.0

    return {
        'total_records' :total,
        'valid_records': valid,
        'invalid_records': invalid,
        'invalid_percentage': invalid_percentage     
    }
    

In [12]:
#  From Day 06 

from datetime import datetime


def is_valid_timestamp(timestamp_str):
    """
        Check if a timestamp string is valid.

        Expected format: YYYY-MM-DD HH:MM
        Returns True if valid, False otherwise.
        """
    if timestamp_str is None:
        return False
    try:
        datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M")
        return True

    except(ValueError, TypeError):
        return False



def validate_energy_record(record):
    """
        Validate a single energy record.

        Returns:
            (True, None) if valid
            (False, error_message) if invalid
        """
    if not isinstance(record, dict):
        return False, "Record must be a dictionary"

    site_id = record.get("site_id")
    if not site_id or not isinstance(site_id, str):
        return False, "Invalid or missing site_id"

    timestamp = record.get("timestamp")
    if not is_valid_timestamp(timestamp):  # Reusing Task 1 method
        return False, "Invalid timestamp format"

    demand = record.get("demand_kw", 0)
    if not isinstance(demand, (int, float)):
        return False, "demand_kw must be numeric"

    if demand < 0:
        return False, "demand_kw must be non-negetive"

    return True, None



def validate_energy_batch(records):
    """
        Validate a list of energy records.

        Returns:
            valid_records: list of valid records
            invalid_records: list of (record, error_message)
        """
    if not isinstance(records, list):
        raise TypeError("records must be a list")

    valid_records = []
    invalid_records = []

    for record in records:
        is_valid, error = validate_energy_record(record)

        if is_valid:
            valid_records.append(record)
        else:
            invalid_records.append((record, error))

    return valid_records, invalid_records

In [14]:
total_records = [
    {"site_id": "S1", "timestamp": "2024-01-01 08:00", "demand_kw": 120},
    {"site_id": None, "timestamp": "2024-01-01 09:00", "demand_kw": 80},
    {"site_id": "S2", "timestamp": "bad_time", "demand_kw": 60},
    {"site_id": "S3", "timestamp": "2024-01-01 10:00", "demand_kw": -40},
    {"site_id": "S1", "timestamp": "2024-01-01 18:00", "demand_kw": 90},
    {"site_id": "S2", "timestamp": "2024-01-01 13:00", "demand_kw": 50},
    {"site_id": "",  "timestamp": "2024-01-01 14:00", "demand_kw": 70},
]
valid_records, invalid_records = validate_energy_batch(total_records)


In [18]:
data_quality_metrics( total_records = total_records  , invalid_records = invalid_records)

{'total_records': 7,
 'valid_records': 3,
 'invalid_records': 4,
 'invalid_percentage': 57.14}